In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\hp\AppData\Local\Temp\ipykernel_27264\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


Note to self: the correct conda env in my system is RAG2

In [4]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Aishi_Tiwari_Resume__Research_ (12).pdf
  ✓ Loaded 1 pages

Processing: intro_pdf.pdf
  ✓ Loaded 2 pages

Total documents loaded: 3


In [5]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)
chunks

Split 3 documents into 15 chunks

Example chunk:
Content: Aishi Tiwari
♂phone+91-9354414404✉aishitiwari1@gmail.com/linkedinLinkedIn
/codeLeetCode Portfolio/terminalGithub
EDUCATION
Guru Gobind Singh Indraprastha University (MAIT)2024 – 2028
B.Tech - Computer...
Metadata: {'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-20T07:42:52+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-20T07:42:52+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdfs\\Aishi_Tiwari_Resume__Research_ (12).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aishi_Tiwari_Resume__Research_ (12).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-20T07:42:52+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-20T07:42:52+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdfs\\Aishi_Tiwari_Resume__Research_ (12).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aishi_Tiwari_Resume__Research_ (12).pdf', 'file_type': 'pdf'}, page_content='Aishi Tiwari\n♂phone+91-9354414404✉aishitiwari1@gmail.com/linkedinLinkedIn\n/codeLeetCode Portfolio/terminalGithub\nEDUCATION\nGuru Gobind Singh Indraprastha University (MAIT)2024 – 2028\nB.Tech - Computer Science and Technology -CGPA: 9.10/10Delhi\nSardar Patel Vidyalaya (Class XII) Completed 2024\nCBSE, Science Stream (PCM) -SCORE: 90.0 Delhi, India\nEXPERIENCE\nIndian Institute of T echnology Delhi with Larsen and T oubroMay 2

In [7]:
pip install numpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2341.50it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\hp\AppData\Local\Temp\ipykernel_27264\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [14]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [11]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.29', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-20T07:42:52+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-20T07:42:52+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.29 (TeX Live 2026) kpathsea version 6.4.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdfs\\Aishi_Tiwari_Resume__Research_ (12).pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Aishi_Tiwari_Resume__Research_ (12).pdf', 'file_type': 'pdf'}, page_content='Aishi Tiwari\n♂phone+91-9354414404✉aishitiwari1@gmail.com/linkedinLinkedIn\n/codeLeetCode Portfolio/terminalGithub\nEDUCATION\nGuru Gobind Singh Indraprastha University (MAIT)2024 – 2028\nB.Tech - Computer Science and Technology -CGPA: 9.10/10Delhi\nSardar Patel Vidyalaya (Class XII) Completed 2024\nCBSE, Science Stream (PCM) -SCORE: 90.0 Delhi, India\nEXPERIENCE\nIndian Institute of T echnology Delhi with Larsen and T oubroMay 2

In [15]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 15 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]

Generated embeddings with shape: (15, 384)
Adding 15 documents to vector store...
Successfully added 15 documents to vector store
Total documents in collection: 15


In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [18]:
rag_retriever

In [30]:
rag_retriever.retrieve("What is Aishi's GPA", top_k=3, score_threshold=float("-inf"))

Retrieving documents for query: 'What is Aishi's GPA'
Top K: 3, Score threshold: -inf
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.13it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_709a6687_6',
  'content': 'Intro\n \non\n \nAishi"\n \n#\n \nAishi\n \n–\n \nAcademic\n \nBackground,\n \nResearch\n \nExperience,\n \nand\n \nCareer\n \nAspirations\n \n \nAishi\n \nis\n \na\n \nthird-year\n \nBachelor\n \nof\n \nTechnology\n \n(B.Tech)\n \nstudent\n \npursuing\n \nComputer\n \nScience\n \nand\n \nTechnology\n \nat\n \nMaharaja\n \nAgrasen\n \nInstitute\n \nof\n \nTechnology\n \n(MAIT),\n \naffiliated\n \nwith\n \nGuru\n \nGobind\n \nSingh\n \nIndraprastha\n \nUniversity\n \n(GGSIPU),\n \nNew\n \nDelhi.\n \nShe\n \nbelongs\n \nto\n \nthe\n \ngraduating\n \nbatch\n \nof\n \n2028\n \nand\n \nhas\n \nmaintained\n \na\n \nstrong\n \nacademic\n \nrecord\n \nwith\n \na\n \nCGPA\n \nof\n \napproximately\n \n9.1\n \nout\n \nof\n \n10.\n \nHer\n \nprimary\n \nacademic\n \ninterests\n \ninclude\n \nArtificial\n \nIntelligence\n \n(AI),\n \nMachine\n \nLearning\n \n(ML),\n \nDeep\n \nLearning,\n \nNatural\n \nLanguage\n \nProcessing\n \n(NLP),\n \nand\n \napplied\n 

In [31]:
queries = [
    "What is Aishi's GPA?",
    "Where does Aishi study?",
    "What research has Aishi done?",
    "What are Aishi's career aspirations?"
]

for query in queries:
    results = rag_retriever.retrieve(
        query,
        top_k=3,
        score_threshold=float("-inf")
    )

    print(f"\nQUERY: {query}")
    print("TOP RESULT:", results[0]["content"][:300] if results else "None")

Retrieving documents for query: 'What is Aishi's GPA?'
Top K: 3, Score threshold: -inf
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.43it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

QUERY: What is Aishi's GPA?
TOP RESULT: Intro
 
on
 
Aishi"
 
#
 
Aishi
 
–
 
Academic
 
Background,
 
Research
 
Experience,
 
and
 
Career
 
Aspirations
 
 
Aishi
 
is
 
a
 
third-year
 
Bachelor
 
of
 
Technology
 
(B.Tech)
 
student
 
pursuing
 
Computer
 
Science
 
and
 
Technology
 
at
 
Maharaja
 
Agrasen
 
Institute
 
of
 
Technol
Retrieving documents for query: 'Where does Aishi study?'
Top K: 3, Score threshold: -inf
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.12it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

QUERY: Where does Aishi study?
TOP RESULT: interests,
 
Aishi
 
enjoys
 
exploring
 
literature,
 
English,
 
music,
 
and
 
subjects
 
outside
 
conventional
 
engineering.
 
She
 
values
 
interdisciplinary
 
learning
 
and
 
is
 
interested
 
in
 
how
 
artificial
 
intelligence
 
can
 
interact
 
with
 
fields
 
such
 
as
 
education,
 

Retrieving documents for query: 'What research has Aishi done?'
Top K: 3, Score threshold: -inf
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.92it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

QUERY: What research has Aishi done?
TOP RESULT: interests,
 
Aishi
 
enjoys
 
exploring
 
literature,
 
English,
 
music,
 
and
 
subjects
 
outside
 
conventional
 
engineering.
 
She
 
values
 
interdisciplinary
 
learning
 
and
 
is
 
interested
 
in
 
how
 
artificial
 
intelligence
 
can
 
interact
 
with
 
fields
 
such
 
as
 
education,
 

Retrieving documents for query: 'What are Aishi's career aspirations?'
Top K: 3, Score threshold: -inf
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

QUERY: What are Aishi's career aspirations?
TOP RESULT: interests,
 
Aishi
 
enjoys
 
exploring
 
literature,
 
English,
 
music,
 
and
 
subjects
 
outside
 
conventional
 
engineering.
 
She
 
values
 
interdisciplinary
 
learning
 
and
 
is
 
interested
 
in
 
how
 
artificial
 
intelligence
 
can
 
interact
 
with
 
fields
 
such
 
as
 
education,
 



In [27]:
query_embedding=embedding_manager.generate_embeddings(["How does Aishi Look?"])[0]

results = vectorstore.collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5,
    include=["documents", "distances", "metadatas"]
)

print("Total stored chunks:", vectorstore.collection.count())
print("Distances:", results["distances"])
print("Documents:", results["documents"])

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 61.25it/s]

Generated embeddings with shape: (1, 384)
Total stored chunks: 15
Distances: [[0.9148836135864258, 0.9404963254928589, 0.9970906972885132, 1.0849943161010742, 1.201754093170166]]
Documents: [["interests,\n \nAishi\n \nenjoys\n \nexploring\n \nliterature,\n \nEnglish,\n \nmusic,\n \nand\n \nsubjects\n \noutside\n \nconventional\n \nengineering.\n \nShe\n \nvalues\n \ninterdisciplinary\n \nlearning\n \nand\n \nis\n \ninterested\n \nin\n \nhow\n \nartificial\n \nintelligence\n \ncan\n \ninteract\n \nwith\n \nfields\n \nsuch\n \nas\n \neducation,\n \npublic\n \npolicy,\n \nand\n \nthe\n \nsocial\n \nsciences.\n \nAlthough\n \nmuch\n \nof\n \nher\n \nacademic\n \nexperience\n \nhas\n \ninvolved\n \ncomputer\n \nvision,\n \nshe\n \nhopes\n \nto\n \nexpand\n \ninto\n \nbroader\n \nmachine\n \nlearning\n \nresearch,\n \nincluding\n \nlanguage\n \nmodels,\n \nintelligent\n \nagents,\n \nand\n \ngeneral-purpose\n \nAI\n \nsystems.\n \n \nLooking\n \nahead,\n \nAishi\n \nplans\n \nto\n \npursue\n